# Strategy Construction & OOS Evaluation

This notebook documents the end-to-end construction of our event-driven, vol-targeted strategy and compares it to a primary-signal baseline over the OOS period **2021-10-21 → 2022-06-29**.

Portfolio construction follows **StrategyWeights lecture (slides 38–43)** exactly:
| Convention | Specification |
|---|---|
| Returns | Simple: $r_{t,k} = (P_{t,k} - P_{t-1,k}) / P_{t-1,k}$ |
| EWMA vol | $\lambda = 2/(\text{span}+1)$; $\mu_t = \lambda r_t + (1-\lambda)\mu_{t-1}$; $\sigma^2_t = \lambda(r_t-\mu_t)^2 + (1-\lambda)\sigma^2_{t-1}$; annualised $\times\sqrt{252}$; floor 2% |
| Weight | $w_{t,k} = \hat{y}_{t,k} \cdot \sigma_{\text{tgt}} / \hat{\sigma}_{t,k}$, $\sigma_{\text{tgt}}=10\%$ |
| Lag | Position decided at $t$, earns $r_{t+1}$ |
| Aggregate | $R^{\text{port}}_{t+1} = \frac{1}{K}\sum_k w_{t,k} r_{t+1,k}$, $K=11$ (flat = cash) |
| Costs | Grinold-Kahn: 2 bps half-spread $+ 10$ bps $\times|\Delta w|$ |

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').is_dir() and (p / 'pyproject.toml').is_file():
            return p
    raise FileNotFoundError('Could not locate stml repo root')

REPO = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO / 'src'))

from stml.io import load_data, load_returns_panel
from stml.new_work import config
from stml.new_work.weights import load_oos_events, make_weights
from stml.new_work.targeting import build_vol_panel
from stml.new_work.vol import ewma_close
from stml.new_work.data import load_oof_probabilities
from stml.experimental.backtest import barrier_backtest, performance_metrics
from stml.experimental.significance import significance_report

EVAL_DIR = REPO / 'results' / 'strategy_eval'

print(f'Repo root : {REPO.name}')
print(f'BOUNDARY  : {config.BOUNDARY.date()}  (GLOBAL_CUT — training cutoff)')
print(f'VOL_METHOD: {config.VOL_METHOD}  span={config.EWMA_SPAN}  floor={config.VOL_FLOOR:.0%}')
print(f'σ_tgt     : {config.SIGMA_TGT:.0%}   max_lev={config.MAX_LEVERAGE}')

## 1. Universe & Data

11 front-month futures across three asset classes.  
The meta-model was trained on OOF events from **2020-01-30 → 2021-10-06** (GLOBAL_CUT).  
A 14-calendar-day embargo separates training from test.

In [ ]:
ohlcv, signals = load_data()
returns_panel  = load_returns_panel(kind='simple')
oos_events     = load_oos_events()
oof_df         = load_oof_probabilities()

# Universe
INSTRUMENTS = sorted(oos_events['instrument'].unique())
ASSET_CLASS = {
    'es1s': 'Equity', 'nq1s': 'Equity', 'fesx1s': 'Equity',
    'cl1s': 'Energy', 'ho1s': 'Energy', 'rb1s': 'Energy', 'ng1s': 'Energy',
    'gc1s': 'Metals', 'si1s': 'Metals', 'hg1s': 'Metals', 'pl1s': 'Metals',
}

summary = (
    oos_events.groupby('instrument')
    .agg(n_events=('ret', 'count'), mean_ret=('ret', 'mean'), frac_long=('side', lambda s: (s > 0).mean()))
    .assign(asset_class=lambda df: df.index.map(ASSET_CLASS))
    [['asset_class', 'n_events', 'mean_ret', 'frac_long']]
    .rename(columns={'asset_class': 'Class', 'n_events': 'OOS events',
                     'mean_ret': 'Mean ret', 'frac_long': 'Frac long'})
)
summary['Mean ret'] = summary['Mean ret'].map('{:.3%}'.format)
summary['Frac long'] = summary['Frac long'].map('{:.0%}'.format)

print(f'OOS period : {oos_events["t_start"].min().date()} → {oos_events["t_start"].max().date()}')
print(f'Total OOS events: {len(oos_events)}  |  Instruments: {len(INSTRUMENTS)}')
print()
print(summary.to_string())

## 2. Strategy Construction

### Step 1 — EWMA Ex-Ante Volatility

For each instrument $k$ and day $t$, we compute an exponentially weighted moving average of **simple** returns using the exact lecture recurrence:

$$\lambda = \frac{2}{\text{span}+1}, \quad
\mu_t = \lambda r_t + (1-\lambda)\mu_{t-1}, \quad
\sigma^2_t = \lambda(r_t - \mu_t)^2 + (1-\lambda)\sigma^2_{t-1}$$

Annualised: $\hat{\sigma}_{t,k} = \sigma_t \times \sqrt{252}$, floored at 2%.  
span = 60 (≈ 3 months), all data ≤ t only (no forward peeking).

### Step 2 — Vol-Targeted Weight

Conviction $\hat{y}_{t,k} \in [-1, +1]$ comes from the primary signal (Benchmark) or meta-model filter (Strategy B).

$$w_{t,k} = \hat{y}_{t,k} \cdot \frac{\sigma_{\text{tgt}}}{\hat{\sigma}_{t,k}}, \qquad \sigma_{\text{tgt}} = 10\%$$

Clipped to $[-10, +10]$ to prevent extreme leverage.

### Step 3 — Portfolio Return (slide 41–44)

Position $w_{t,k}$ is decided from data $\leq t$ and earns the **next-day** simple return $r_{t+1,k}$:

$$R^{\text{port}}_{t+1} = \frac{1}{K} \sum_{k=1}^{K} w_{t,k}\, r_{t+1,k}$$

$K = 11$ always — instruments sitting flat contribute 0 to the numerator but remain in the denominator (cash allocation).

Net of transaction costs: $R^{\text{net}}_{t+1} = R^{\text{port}}_{t+1} - \frac{1}{K}\sum_k c_k |\Delta w_{t,k}|$

In [ ]:
# Illustrate EWMA vol for three instruments over the full sample
DEMO_INSTS = ['cl1s', 'es1s', 'gc1s']
DEMO_NAMES = {'cl1s': 'Crude Oil (CL)', 'es1s': 'S&P 500 (ES)', 'gc1s': 'Gold (GC)'}

fig, ax = plt.subplots(figsize=(10, 3.5))

for inst in DEMO_INSTS:
    sub = ohlcv[ohlcv['instrument'] == inst].set_index('date').sort_index()
    vol = ewma_close(sub['close'], span=config.EWMA_SPAN)
    ax.plot(vol.index, vol * 100, label=DEMO_NAMES[inst], linewidth=1.2)

# Shade OOS period
oos_start = pd.Timestamp('2021-10-21')
oos_end   = pd.Timestamp('2022-06-29')
ax.axvspan(oos_start, oos_end, alpha=0.12, color='steelblue', label='OOS period')
ax.axvline(config.BOUNDARY, color='black', linestyle='--', linewidth=0.8, label=f'GLOBAL_CUT ({config.BOUNDARY.date()})')

ax.set_ylabel('Annualised vol (%)')
ax.set_title('EWMA(60) Annualised Volatility — lecture recurrence')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 3. Meta-Model Filter (Method B)

The meta-model (trained via CPCV on OOF events) outputs a **calibrated probability** $\hat{p} \in (0, 1)$ that the primary signal is worth following for each event.

**All-or-nothing filter** (Method B-aon): follow the primary signal only when $\hat{p} > 0.5$:
$$\hat{y}_{t,k} = \text{side}_{t,k} \cdot \mathbf{1}[\hat{p}_{t,k} > 0.5]$$

This reduces the event count from 1373 → 745 (54%), skipping events where the meta-model has low confidence.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

# Left: calibrated probability histogram for OOS events
p_hat = oos_events['calibrated_proba'].dropna()
axes[0].hist(p_hat, bins=30, color='steelblue', edgecolor='white', linewidth=0.4)
axes[0].axvline(0.5, color='crimson', linestyle='--', linewidth=1, label='Threshold')
axes[0].set_xlabel('Calibrated probability $\\hat{p}$')
axes[0].set_ylabel('Event count')
axes[0].set_title('OOS Meta-Model Probabilities')
axes[0].legend()

taken = (p_hat > 0.5).sum()
total = len(p_hat)
axes[0].text(0.03, 0.92, f'Taken: {taken}/{total} ({taken/total:.0%})',
             transform=axes[0].transAxes, fontsize=9)

# Right: event returns split by meta decision
evts = oos_events.dropna(subset=['calibrated_proba']).copy()
evts['taken'] = evts['calibrated_proba'] > 0.5
for taken_flag, label, color in [(True, 'Taken (p̂>0.5)', 'steelblue'), (False, 'Skipped (p̂≤0.5)', 'lightcoral')]:
    sub = evts[evts['taken'] == taken_flag]['ret']
    axes[1].hist(sub, bins=30, alpha=0.6, color=color, label=f'{label} (n={len(sub)})', edgecolor='white', linewidth=0.3)
axes[1].set_xlabel('Event return')
axes[1].set_ylabel('Count')
axes[1].set_title('Return Distribution by Meta-Model Decision')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Mean returns by decision
print('Mean event return by meta-model decision:')
print(evts.groupby('taken')['ret'].mean().rename({True: 'Taken (p̂>0.5)', False: 'Skipped (p̂≤0.5)'}).map('{:.4%}'.format).to_string())

## 4. OOS Evaluation

We compare two methods:

| Method | Conviction $\hat{y}$ | Events | Description |
|---|---|---|---|
| **A — Benchmark** | $\text{side}$ | All 1373 | Trade every event at full primary-signal conviction |
| **B — Meta-filtered** | $\text{side} \cdot \mathbf{1}[\hat{p}>0.5]$ | 745 (54%) | Only trade when meta-model is confident |

Both methods use the same vol-targeting, portfolio construction, and cost model.

In [ ]:
# Build shared inputs once
vol_panel = build_vol_panel(ohlcv, method=config.VOL_METHOD,
                            window=config.LOOKBACK_L, span=config.EWMA_SPAN)

reports = {}
for method in ['A', 'B-aon']:
    events, weights = make_weights(
        method,
        oos_events=oos_events,
        vol_panel=vol_panel,
    )
    reports[method] = barrier_backtest(events, weights, returns_panel)
    m = reports[method].metrics
    n_active = (weights.abs() > 0).sum()
    label = {'A': 'A Benchmark', 'B-aon': 'B Meta-filtered'}[method]
    print(f'{label}: Sharpe={m["sharpe"]:.3f}  ann_ret={m["ann_return"]:+.1%}  '
          f'ann_vol={m["ann_vol"]:.1%}  max_DD={m["max_dd"]:.1%}  '
          f'events={n_active}/1373')

In [ ]:
# ── Cumulative returns plot ──────────────────────────────────────────────────
COLORS  = {'A': '#4878CF', 'B-aon': '#D65F5F'}
LABELS  = {'A': 'A — Benchmark (primary signal)', 'B-aon': 'B — Meta-filtered (p̂ > 0.5)'}
METHODS = ['A', 'B-aon']

fig, (ax_cum, ax_dd) = plt.subplots(2, 1, figsize=(11, 6),
                                     gridspec_kw={'height_ratios': [3, 1]},
                                     sharex=True)

for method in METHODS:
    r = reports[method].net_returns
    cum = (1 + r).cumprod() - 1
    ax_cum.plot(cum.index, cum * 100, label=LABELS[method],
                color=COLORS[method], linewidth=1.6)

    # Drawdown
    eq  = (1 + r).cumprod()
    dd  = (eq / eq.cummax() - 1) * 100
    ax_dd.fill_between(dd.index, dd, 0, alpha=0.35, color=COLORS[method])
    ax_dd.plot(dd.index, dd, color=COLORS[method], linewidth=0.8)

ax_cum.axhline(0, color='black', linewidth=0.6, linestyle='--')
ax_cum.set_ylabel('Cumulative net return (%)')
ax_cum.set_title('OOS Cumulative Net Returns  (2021-10-21 → 2022-06-29)',
                  fontsize=11, fontweight='bold')
ax_cum.legend(fontsize=9)
ax_cum.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

ax_dd.set_ylabel('Drawdown (%)')
ax_dd.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax_dd.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax_dd.xaxis.set_major_locator(mdates.MonthLocator(interval=2))

# Annotate final returns
for method in METHODS:
    r   = reports[method].net_returns
    cum = (1 + r).cumprod() - 1
    ax_cum.annotate(
        f"{cum.iloc[-1]*100:+.1f}%",
        xy=(cum.index[-1], cum.iloc[-1] * 100),
        xytext=(8, 0), textcoords='offset points',
        color=COLORS[method], fontsize=9, va='center',
    )

fig.align_ylabels()
plt.tight_layout()

fig_path = EVAL_DIR / 'cumulative_returns.png'
EVAL_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {fig_path.relative_to(REPO)}')

In [ ]:
# ── Performance metrics table ────────────────────────────────────────────────
rows = []
for method in METHODS:
    r   = reports[method].net_returns
    m   = reports[method].metrics
    sr  = significance_report(r.values, n_boot=2000, seed=42)
    w   = reports[method].weights
    n_active = (w.fillna(0).abs().sum(axis=1) > 0).sum()
    rows.append({
        'Method':          LABELS[method],
        'Ann return':      f"{m['ann_return']:+.2%}",
        'Ann vol':         f"{m['ann_vol']:.2%}",
        'Sharpe':          f"{m['sharpe']:.3f}",
        't-stat':          f"{sr.t_stat:.2f}",
        'SR 95% CI':       f"[{sr.bootstrap_ci_low*252**0.5:.2f}, {sr.bootstrap_ci_high*252**0.5:.2f}]",
        'PSR(SR*=0)':      f"{sr.psr_zero:.3f}",
        'Max DD':          f"{m['max_dd']:.2%}",
        'Total cost (bps)': f"{m['total_cost_bps']:.0f}",
        'Ann turnover':    f"{m['turnover_per_year']:.2f}",
        'Events taken':    f"{(w.fillna(0).abs() > 0).any(axis=1).sum()}/180 days",
    })

perf = pd.DataFrame(rows).set_index('Method').T
print(perf.to_string())

## 5. Interpretation

**Benchmark (A):** Trades every event the primary signal fires on.  
Sharpe ≈ 1.41, annualised return ≈ +8.6%, vol ≈ 6.1%, max DD ≈ −3.4%.

**Meta-filtered strategy (B):** Applies the meta-model's binary filter — skips events where $\hat{p} \leq 0.5$.  
Sharpe ≈ 1.71 (+0.30 vs benchmark), lower absolute return (+4.9%) but also lower vol (2.9%).  
The Sharpe improvement arises from skipping low-quality trades, halving the number of events (745 vs 1373) and cutting max DD from −3.4% to −1.4%.

**Costs:** Grinold-Kahn model (2 bps spread + 10 bps impact). At 54% event selection, transaction costs halve relative to the benchmark (188 vs 416 bps total), and the breakeven half-spread improves from 17.8 → 22.5 bps.

**Statistical significance:** Both methods have Sharpe t-stats ≈ 1.2–1.4 over 180 OOS trading days — not yet conventionally significant at 5%, but PSR(SR*=0) ≈ 0.86–0.91 indicates meaningful evidence of positive Sharpe. Confidence intervals are wide given the short OOS window (180 days).

**Leakage controls:** Single fitting boundary at GLOBAL_CUT (2021-10-06). 14-calendar-day embargo enforced. Hidden test set (Jul–Dec 2022) never accessed. EWMA vol updates causally through OOS; only SOPS and neural weights are frozen at BOUNDARY.

In [ ]:
# ── Weight panel: active days by instrument ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)

for ax, method in zip(axes, METHODS):
    w = reports[method].weights
    # Mean absolute weight per instrument (proxy for risk allocation)
    mean_w = w.abs().mean().sort_values(ascending=True)
    colors_bar = ['#4878CF' if ASSET_CLASS.get(i) == 'Equity'
                  else '#D65F5F' if ASSET_CLASS.get(i) == 'Energy'
                  else '#6ACC65' for i in mean_w.index]
    bars = ax.barh(mean_w.index, mean_w.values, color=colors_bar, edgecolor='white', linewidth=0.4)
    ax.set_xlabel('Mean |weight|')
    ax.set_title(LABELS[method], fontsize=9)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#4878CF', label='Equity'),
                   Patch(facecolor='#D65F5F', label='Energy'),
                   Patch(facecolor='#6ACC65', label='Metals')]
axes[1].legend(handles=legend_elements, fontsize=8, loc='lower right')

fig.suptitle('Mean Absolute Weight by Instrument (OOS period)', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Summary

| | Benchmark (A) | Meta-filtered (B) |
|---|---|---|
| Conviction | $\hat{y} = \text{side}$ | $\hat{y} = \text{side}\cdot\mathbf{1}[\hat{p}>0.5]$ |
| Events traded | 1373 / 1373 | 745 / 1373 |
| OOS Sharpe | 1.41 | **1.71** |
| Ann return (net) | +8.6% | +4.9% |
| Ann vol | 6.1% | 2.9% |
| Max drawdown | −3.4% | −1.4% |
| Total costs | 416 bps | 188 bps |

The meta-model filter improves risk-adjusted performance (+0.30 Sharpe) by selectively sitting out the lower-confidence half of events, while halving transaction costs and maximum drawdown.  
The tradeoff is lower absolute return — the filtered strategy earns 4.9% p.a. vs 8.6% for the benchmark, reflecting the cash-like return on the 46% of days it sits flat.